# Testing your fine-tuned model 

Let's load and run the fine-tuned model.  

Using the model to generate output (as opposed to fine-tuning or creating the model) is known as 'inference'. 

You've seen similar code in the first notebook so this should be familiar to you now 😊

## Configuration

In [ ]:
import torch
from typing import Any
import yaml


MODEL_NAME = "microsoft/Phi-3.5-mini-instruct"

def get_device():
    """Determine the available device for computation."""
    if torch.cuda.is_available():
        return "cuda"
    elif torch.backends.mps.is_available(): 
        return "mps"
    return "cpu"

DEVICE = get_device()

# Load configuration parameters from a YAML file into a dictionary.
# This file typically contains parameters such as learning rates, batch sizes, model names, etc., used for fine-tuning.
with open("./finetuning_params.yml", "r") as file:
    config: dict[str, Any] = yaml.safe_load(file)

print("Config loaded")

## Loading the model and tokenizer

Choose the option which applies to you: 

#### Option 1: If you have run finetuning

Run the cell below to load your finetuned model. 

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

output_dir = config["final_model_dir"]  # where the model is saved
tokenizer = AutoTokenizer.from_pretrained(output_dir)
model = AutoModelForCausalLM.from_pretrained(output_dir).to(DEVICE)

#### Option 2: If you want to test out a fully finetuned model

Load a model which has been finetuned for ~120 epochs on the full dataset (3997 samples). 
Note that its output on the test dataset will be used in the next evaluation lab.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("quyvu/finetuned_fn_calling_ai_learning_day").to(DEVICE)
tokenizer = AutoTokenizer.from_pretrained("quyvu/finetuned_fn_calling_ai_learning_day")

## Preparing the prompt

You may recall this system prompt from the previous labs - it sets out the instructions for the model to generate the desired output.

In [ ]:
from textwrap import dedent

# Define a system prompt for an in-car virtual assistant.
# This prompt guides the behavior of the assistant by specifying how it should respond to user inputs.
SYSTEM_PROMPT = dedent(
    """
    You are an in car virtual assistant that maps user's inputs to the corresponding function call in the vehicle. You must respond with only a JSON object matching the following schema:

    {
        "function_name": <name of the function>,
        "parameters": {
            <arg1_name>: <arg1_value>,
            <arg2_name>: <arg2_value>,
            ...
        }
    }

    Important: DO NOT include any other information in the response.
    """
)

## Inference

The time has come! Let's test out our model.

The block of code below is quite scary, but as a wise man once said: "don't worry about it if you don't understand". Explanation is provided under the name of each component to guide your understanding.

Or you can just run the code and see the magic happen!🪄

In [ ]:
import time
import json
from tqdm import tqdm
from transformers import LogitsProcessor, LogitsProcessorList


class FirstTokenLogitsProcessor(LogitsProcessor):
    """
    This is used to capture the time when the first token is generated, which will
    eventually be used to compute time to first token
    """
    def __init__(self):
        self.first_token_time = (
            None  # Store the timestamp when first token is generated
        )

    def __call__(self, input_ids: torch.Tensor, scores: torch.Tensor) -> torch.Tensor:
        if self.first_token_time is None:
            self.first_token_time = (
                time.time()
            )  # Capture time of first token generation
        return scores  # Return unchanged scores to allow generation to continue


# Post-processing function: formats the model output
def postprocess_output(raw_output: str) -> dict[str, Any]:
    """
    The model gives us the answer in a blob of string. This converts the model output
    into a structured format as follows

    {
        "function_name": <function_name>,
        "parameters": {
            "parameter_1": <value_1>,
            "parameter_2": <value_2>,
            ...
        }
    }

    which can be used for downstream application and analysis.
    """
    try:
        parsed_output = json.loads(
            raw_output
        )  # Assuming the model returns JSON-like text
        if isinstance(parsed_output, dict) and "function_name" in parsed_output:
            return parsed_output  # Valid structured output
    except json.JSONDecodeError:
        pass

    # Default fallback if model response is not well-formed
    return {
        "function_name": "unknown_function",
        "parameters": {"raw_response": raw_output},
    }


# Function to calculate metrics
def calculate_metrics(
    prompt_tokens: int,
    completion_tokens: int,
    time_to_first_token: float,
    total_time: float,
) -> dict[str, float]:
    """
    Calculates performance metrics, such as latency, number of tokens generated, and
    throughput. It does not tell us about how accurate the model is - this is what we'll
    cover in the next lab!
    """
    return {
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": prompt_tokens + completion_tokens,
        "time_to_first_token_seconds": round(time_to_first_token, 4),  # Time to first token in seconds
        "total_time_seconds": round(total_time, 4),  # Total inference time in seconds
        "tokens_per_second": round(completion_tokens / total_time, 4) if total_time > 0 else 0.0,
    }


def run_single_inference(
    query: str,
    model,
    tokenizer,
    system_prompt: str,
) -> tuple[dict[str, Any], dict[str, float]]:
    """
    This feeds the user's query to the model, and gives us the well formatted output,
    along with the performance metrics calculated in the `calculate_metrics` function.
    """

    messages = [
        {"role": "system", "content": system_prompt},  # System instruction
        {"role": "user", "content": query},  # User input
    ]

    # Prepare input for the model
    input_ids = tokenizer.apply_chat_template(
        messages, truncation=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    prompt_tokens = input_ids.shape[1]

    # Initialize logits processor to capture first token time
    first_token_processor = FirstTokenLogitsProcessor()

    # Start timing for total time capture of request
    start_time = time.time()

    completion_tokens = model.generate(
        input_ids=input_ids,
        max_new_tokens=256,
        do_sample=False,
        top_k=50,
        logits_processor=LogitsProcessorList([first_token_processor]),
    )

    # End timing
    # Capture first token time
    time_to_first_token = (
        first_token_processor.first_token_time - start_time
        if first_token_processor.first_token_time
        else None
    )
    # Capture total time taken
    total_time = time.time() - start_time  # Total inference time

    # calculate completion tokens
    num_completion_tokens = len(completion_tokens[0])

    raw_output = tokenizer.decode(
        completion_tokens[0][input_ids.shape[1] :],
        skip_special_tokens=True,
    )

    formatted_output = postprocess_output(raw_output)
    metrics = calculate_metrics(
        prompt_tokens, num_completion_tokens, time_to_first_token, total_time
    )

    return formatted_output, metrics


def run_inference(queries: list[str], model, tokenizer, system_prompt: str) -> list[tuple[dict[str, Any], dict[str, float]]]:
    """
    This just runs the `run_single_inference` function for multiple queries and
    returns the results in a list, so you don't have to keep calling the function
    multiple times.
    """
    results = []

    for query in tqdm(queries, desc="Running inference"):
        formatted_output, metrics = run_single_inference(query, model, tokenizer, system_prompt)
        results.append((formatted_output, metrics))

    return results

### Test out the model on your own prompts

In [ ]:
# Replace the user query with the query you want to test
user_query = "Turn on the AC"

response, metrics = run_single_inference(user_query, model, tokenizer, SYSTEM_PROMPT)

print(f"--- User Query: {user_query}")
print(f"--- Model Response:\n{json.dumps(response, indent=4)}")
print(f"--- Metrics:\n{json.dumps(metrics, indent=4)}")


### If you want to run multiple prompts in one go, run the cell below:

In [ ]:
# Replace the user queries with the queries you want to test
user_queries = [
    "turn up the volume to 45",
    "set the temperature to 72 degrees",
]

outputs = run_inference(user_queries, model, tokenizer, SYSTEM_PROMPT)
for user_query, (response, metrics) in zip(user_queries, outputs):
    print(f"--- User Query: {user_query}")
    print(f"--- Model Response:\n{json.dumps(response, indent=4)}")
    print(f"--- Metrics:\n{json.dumps(metrics, indent=4)}")
    print("--------------------------------")

<span style="color: green;">**QUESTION**</span>: What do you think about the model's output? Does it make sense? How else would you test the model?

<details>
    <summary>Click to show the answer</summary>

Some ideas you may want to try:
- What if you feed it something it has never seen before?
- What if you ask it to make a pizza?
- What if you ask it to set the temperature to 1000000000000 degrees?
- What if you talk to it like a drunk pirate?

Think about how the model will be used in a car. How would the user interact with it? What would be their expectations? What about responsible AI?

</details>


<span style="color: green;">**QUESTION**</span>: How would you test the model more hollistically? Would it make sense to run the model a bunch of times and visually inspect how good the output is?

<details>
    <summary>Click to show the answer</summary>

This will be covered in the next lab! But remember that, qualitative evaluation is just as important as quantitative evaluation. There are nuances that cannot be captured by metrics alone.
</details>
